<a href="https://colab.research.google.com/github/Jeremiah004/Jeremiah004/blob/main/TB_HIV_clean_analysis_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TB blood gene signatures and HIV status: clean analysis script

One notebook that regenerates every number in the paper outline. Run the cells **top to bottom, once**, in a fresh R session (Colab, R runtime).

Rules for this notebook
- Do not ask the Colab AI to edit or "fix" these cells. Earlier, it silently replaced our signature lists with its own. If a cell fails, copy the error and bring it back to the chat.
- Every step stops with an error if something is missing. There are no silent fallbacks.
- Results are saved to a `results/` folder as CSV files at the end.
- Settings are in the first code cell (`B` = number of bootstrap resamples; earlier exploratory runs used 500).

In [1]:
## 0. SETTINGS AND PACKAGES ---------------------------------------------
SEED <- 1
B <- 1000                 # bootstrap resamples
N_GENES <- 3000           # most variable genes used by the elastic-net models

# Packages go on the Colab machine's LOCAL disk (installing into Google Drive failed before).
# They are lost when the session ends, so a new session needs this cell again.
LOCAL_LIB <- "/content/R_local"
dir.create(LOCAL_LIB, showWarnings = FALSE, recursive = TRUE)
.libPaths(c(LOCAL_LIB, .libPaths()))

if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", lib = LOCAL_LIB)
have <- function(p) requireNamespace(p, quietly = TRUE)

bioc_needed <- c("curatedTBData", "MultiAssayExperiment", "SummarizedExperiment")
bioc_missing <- bioc_needed[!vapply(bioc_needed, have, logical(1))]
if (length(bioc_missing) > 0) {
  BiocManager::install(bioc_missing, lib = LOCAL_LIB, Ncpus = 4, update = FALSE, ask = FALSE)
}
if (!have("glmnet")) install.packages("glmnet", lib = LOCAL_LIB, Ncpus = 4)

# Stop early, with a clear message, if anything failed to install
required <- c(bioc_needed, "glmnet")
still_missing <- required[!vapply(required, have, logical(1))]
if (length(still_missing) > 0) {
  stop("These packages failed to install: ", paste(still_missing, collapse = ", "),
       ". Scroll up, find the first line that contains 'ERROR', and copy it into the chat.")
}

suppressPackageStartupMessages({
  library(curatedTBData)
  library(MultiAssayExperiment)
  library(SummarizedExperiment)
  library(glmnet)
})

# The signature list is read from the TBSignatureProfiler SOURCE file (no compiling, no dependencies).
if (!file.exists("TBSignatureProfiler/data/TBsignatures.rda")) {
  tarball <- download.packages("TBSignatureProfiler", destdir = ".",
                               repos = BiocManager::repositories(), type = "source")
  untar(tarball[1, 2])
}
load("TBSignatureProfiler/data/TBsignatures.rda")
stopifnot(exists("TBsignatures"), length(TBsignatures) > 0)
cat("All packages loaded. Signatures in TBsignatures:", length(TBsignatures), "\n")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.1 (2026-06-24)

Installing package(s) 'BiocVersion', 'curatedTBData', 'MultiAssayExperiment',
  'SummarizedExperiment'

also installing the dependencies ‘png’, ‘Biostrings’, ‘filelock’, ‘KEGGREST’, ‘XVector’, ‘BiocFileCache’, ‘RSQLite’, ‘AnnotationDbi’, ‘SparseArray’, ‘matrixStats’, ‘abind’, ‘AnnotationHub’, ‘ExperimentHub’, ‘Biobase’, ‘BiocBaseUtils’, ‘BiocGenerics’, ‘DelayedArray’, ‘GenomicRanges’, ‘IRanges’, ‘MatrixGenerics’, ‘S4Vectors’, ‘Seqinfo’, ‘S4Arrays’


also installing the dependencies ‘iterators’, ‘foreach’, ‘shape’, ‘RcppEigen’


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com



All packages loaded. Signatures in TBsignatures: 80 


In [2]:
## 1. HELPER FUNCTIONS ------------------------------------------------------
# Fast AUC (Mann-Whitney form). y = 0/1, s = score (higher = more likely TB).
auc_fast <- function(y, s) {
  r <- rank(s); n1 <- sum(y == 1); n0 <- sum(y == 0)
  (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}

# Bootstrap the AUC gap (HIV-negative minus HIV-positive). neg/pos = row indices.
boot_gap <- function(y, s, neg, pos, B) {
  replicate(B, {
    i <- sample(neg, replace = TRUE); j <- sample(pos, replace = TRUE)
    auc_fast(y[i], s[i]) - auc_fast(y[j], s[j])
  })
}

q95 <- function(d) quantile(d, c(0.025, 0.5, 0.975), names = FALSE)   # low, median, high

# AUC gap of a prediction vector (used for the models)
drop_of <- function(yt, pr, h) {
  auc_fast(yt[h == "Negative"], pr[h == "Negative"]) -
    auc_fast(yt[h == "Positive"], pr[h == "Positive"])
}

In [3]:
## 2. LOAD GSE37250 -----------------------------------------------------------
d0   <- curatedTBData("GSE37250", dry.run = FALSE, curated.only = TRUE)
mae  <- d0[["GSE37250"]]
expr <- assay(mae[["assay_curated"]])
meta <- as.data.frame(colData(mae))

stopifnot(all(colnames(expr) == rownames(meta)),
          all(c("GBP5", "DUSP3", "KLF2") %in% rownames(expr)),
          sum(is.na(expr)) == 0)

tb      <- meta$TBStatus              # PTB / LTBI / OD
hiv     <- meta$HIVStatus             # Negative / Positive
country <- meta$GeographicalRegion    # Malawi / South Africa
is_tb   <- as.numeric(tb == "PTB")
z       <- t(scale(t(expr)))          # every gene on the same scale

cat("Genes x people:", dim(expr), "| value range:", round(range(expr), 1), "\n")
print(table(TB = tb, HIV = hiv, useNA = "ifany"))
print(table(Country = country, HIV = hiv, TB = tb))

curated.only = TRUE. Download curated version. 
Set curated.only = FALSE if want to download both raw and curated data.

Downloading: GSE37250



  |=======================                                               |  33%
  |===============================================                       |  67%
  |======================================================================| 100%



Finished!



Genes x people: 31537 537 | value range: 4.8 15.8 
      HIV
TB     Negative Positive
  LTBI       83       84
  OD         83       92
  PTB        97       98
, , TB = LTBI

              HIV
Country        Negative Positive
  Malawi             35       36
  South Africa       48       48

, , TB = OD

              HIV
Country        Negative Positive
  Malawi             34       30
  South Africa       49       62

, , TB = PTB

              HIV
Country        Negative Positive
  Malawi             51       51
  South Africa       46       47



In [4]:
## 3. SUPPLEMENTARY PILOT: 3-gene Sweeney score, TB vs everyone else ---------
# This first pilot mixed LTBI and other disease in the comparison group.
# It is kept only as a supplementary analysis (see section 4 for the proper task split).
sweeney <- (expr["GBP5", ] + expr["DUSP3", ]) / 2 - expr["KLF2", ]

cat("AUC everyone:    ", round(auc_fast(is_tb, sweeney), 3), "\n")
for (h in c("Negative", "Positive")) {
  k <- hiv == h
  cat("AUC HIV-", tolower(h), ":  ", round(auc_fast(is_tb[k], sweeney[k]), 3), "\n", sep = "")
}
for (h in c("Negative", "Positive")) for (cn in unique(country)) {
  k <- hiv == h & country == cn
  cat("AUC", h, "-", cn, ":", round(auc_fast(is_tb[k], sweeney[k]), 3), "(n =", sum(k), ")\n")
}

set.seed(SEED)
gap_report <- function(label, neg, pos) {
  q <- q95(boot_gap(is_tb, sweeney, neg, pos, B))
  cat(sprintf("Gap %-13s median %.3f (95%% interval %.3f to %.3f)\n", label, q[2], q[1], q[3]))
}
gap_report("All", which(hiv == "Negative"), which(hiv == "Positive"))
for (cn in unique(country)) {
  gap_report(cn, which(hiv == "Negative" & country == cn), which(hiv == "Positive" & country == cn))
}

AUC everyone:     0.897 
AUC HIV-negative:  0.923
AUC HIV-positive:  0.877
AUC Negative - Malawi : 0.895 (n = 120 )
AUC Negative - South Africa : 0.946 (n = 143 )
AUC Positive - Malawi : 0.867 (n = 117 )
AUC Positive - South Africa : 0.872 (n = 157 )
Gap All           median 0.047 (95% interval -0.003 to 0.100)
Gap Malawi        median 0.030 (95% interval -0.052 to 0.121)
Gap South Africa  median 0.075 (95% interval -0.000 to 0.151)


In [5]:
## 4. SIGNATURES AND THE MAIN BENCHMARK ------------------------------------------
# Signature lists come ONLY from the TBSignatureProfiler package object.
# Removed: risk / response / failure / pneumonia signatures, the Kaforou signatures
# (built on this cohort) and one duplicate (Gliddon_2_OD_4).
sigs <- TBsignatures[!grepl("RISK|RES|FAIL|PNA|Kaforou|Gliddon_2_OD_4", names(TBsignatures))]
cat("Signatures kept after exclusions:", length(sigs), "of", length(TBsignatures), "\n")

# Each signature is scored on the comparison it was built for (read from its name):
sig_task <- function(s) {
  if (grepl("_OD_", s)) "TB vs OD" else if (grepl("_HIV_", s)) "TB vs rest" else "TB vs LTBI"
}
task_idx <- function(task) {
  switch(task,
         "TB vs OD"   = tb %in% c("PTB", "OD"),
         "TB vs LTBI" = tb %in% c("PTB", "LTBI"),
         "TB vs rest" = rep(TRUE, length(tb)))
}

# Score = mean of directed z-scores. Direction of each gene (up or down in TB) is learned in
# the OTHER country only, then applied to the test country.
score_cross <- function(g, use_i) {
  sc <- rep(NA_real_, ncol(z))
  for (cn in unique(country)) {
    test  <- country == cn
    train <- use_i & country != cn
    dd <- rowMeans(z[g, train & is_tb == 1, drop = FALSE]) -
          rowMeans(z[g, train & is_tb == 0, drop = FALSE])
    sc[test] <- colMeans(z[g, test, drop = FALSE] * sign(dd))
  }
  sc
}

# Coverage rule used everywhere: at least 3 genes and at least 80 percent of the signature
usable_genes <- function(s, rows) {
  g <- intersect(sigs[[s]], rows)
  if (length(g) < 3 || length(g) < 0.8 * length(sigs[[s]])) return(NULL)
  g
}

set.seed(SEED)
bench <- do.call(rbind, lapply(names(sigs), function(s) {
  g <- usable_genes(s, rownames(z)); if (is.null(g)) return(NULL)
  task  <- sig_task(s); use_i <- task_idx(task)
  sc    <- score_cross(g, use_i)
  neg   <- which(use_i & hiv == "Negative"); pos <- which(use_i & hiv == "Positive")
  a_n   <- auc_fast(is_tb[neg], sc[neg]);    a_p <- auc_fast(is_tb[pos], sc[pos])
  d     <- boot_gap(is_tb, sc, neg, pos, B); q <- q95(d)
  data.frame(Signature = s, Task = task, Genes = paste0(length(g), "/", length(sigs[[s]])),
             AUC_Neg = round(a_n, 3), AUC_Pos = round(a_p, 3), Gap = round(a_n - a_p, 3),
             CI_low = round(q[1], 3), CI_high = round(q[3], 3),
             p_boot = max(1 / B, min(1, 2 * min(mean(d <= 0), mean(d >= 0)))))
}))
bench$p_adj      <- p.adjust(bench$p_boot, "BH")   # signatures overlap in genes: not independent tests
bench$Clear_gap  <- bench$CI_low > 0
print(bench[order(-bench$Gap), ], row.names = FALSE)

cat("\nSignatures scored:", nrow(bench), "\n")
print(table(Task = bench$Task, Clear_gap = bench$Clear_gap))
cat("Clear after Benjamini-Hochberg (p_adj < 0.05):\n")
print(table(Task = bench$Task, Clear_after_BH = bench$p_adj < 0.05))

Signatures kept after exclusions: 59 of 80 
         Signature       Task   Genes AUC_Neg AUC_Pos    Gap CI_low CI_high
    Sivakumaran_11 TB vs LTBI   11/11   0.931   0.811  0.120  0.053   0.196
       Zhao_NANO_6 TB vs LTBI     6/6   0.918   0.803  0.116  0.045   0.192
             Roe_3 TB vs LTBI     3/3   0.937   0.824  0.113  0.046   0.185
            Chen_5 TB vs LTBI     5/5   0.894   0.785  0.108  0.029   0.198
           Gjoen_7 TB vs LTBI     7/7   0.932   0.833  0.099  0.031   0.170
            Kaul_3 TB vs LTBI     3/3   0.956   0.857  0.099  0.040   0.160
        Jacobsen_3 TB vs LTBI     3/3   0.927   0.840  0.087  0.017   0.147
          Leong_24 TB vs LTBI   22/24   0.947   0.861  0.086  0.026   0.148
       Anderson_42 TB vs LTBI   32/37   0.941   0.856  0.085  0.025   0.147
       Natarajan_7 TB vs LTBI     7/7   0.953   0.869  0.084  0.026   0.137
     Gliddon_HIV_3 TB vs rest     3/3   0.856   0.775  0.081  0.012   0.151
              Li_3 TB vs LTBI     3/3   0.96

In [6]:
## 5. LTBI-ONLY CHECK: do HIV-positive people WITHOUT active TB score higher? -----------------
ltbi_i   <- tb == "LTBI"
ltbi_sig <- names(sigs)[vapply(names(sigs), sig_task, character(1)) == "TB vs LTBI"]

ltbi_tab <- do.call(rbind, lapply(ltbi_sig, function(s) {
  g <- usable_genes(s, rownames(z)); if (is.null(g)) return(NULL)
  sc <- score_cross(g, task_idx("TB vs LTBI"))
  a <- sc[ltbi_i & hiv == "Positive"]; b <- sc[ltbi_i & hiv == "Negative"]
  data.frame(Signature = s, Median_Neg = round(median(b), 2), Median_Pos = round(median(a), 2),
             p_raw = wilcox.test(a, b)$p.value)
}))
ltbi_tab$p_adj         <- p.adjust(ltbi_tab$p_raw, "BH")
ltbi_tab$Higher_in_HIV <- ltbi_tab$Median_Pos > ltbi_tab$Median_Neg
# Earlier runs printed some p-values as exactly 0. Report extremely small values as "<1e-15".
ltbi_tab$p_adj_report  <- ifelse(ltbi_tab$p_adj < 1e-15, "<1e-15", format(signif(ltbi_tab$p_adj, 2)))
print(ltbi_tab[order(ltbi_tab$p_adj), c("Signature", "Median_Neg", "Median_Pos", "p_adj_report", "Higher_in_HIV")],
      row.names = FALSE)
cat("\nSignatures:", nrow(ltbi_tab),
    "| HIV-positive LTBI scores higher in", sum(ltbi_tab$Higher_in_HIV),
    "| significant after correction:", sum(ltbi_tab$p_adj < 0.05), "\n")

      Signature Median_Neg Median_Pos p_adj_report Higher_in_HIV
         Chen_5      -0.79       0.17       <1e-15          TRUE
         Kaul_3      -1.18      -0.04       <1e-15          TRUE
   Maertzdorf_4      -1.14      -0.13       <1e-15          TRUE
          Roe_3      -0.94      -0.13       <1e-15          TRUE
      Esmail_82      -0.43      -0.13      2.5e-15          TRUE
     Esmail_203      -0.49      -0.18      4.4e-14          TRUE
      Vargas_18      -0.96      -0.35      4.4e-14          TRUE
      Vargas_42      -0.97      -0.35      1.3e-13          TRUE
    Natarajan_7      -1.14      -0.32      2.1e-13          TRUE
  Maertzdorf_15      -0.93      -0.38      6.4e-13          TRUE
       Gjoen_10      -0.55      -0.09      2.8e-12          TRUE
      Berry_393      -0.81      -0.34      4.8e-12          TRUE
     Blankley_5      -1.09      -0.61      1.7e-11          TRUE
    Anderson_42      -0.70      -0.30      6.6e-11          TRUE
        Gjoen_7      -0.8

In [7]:
## 6. CIRCULARITY: which signatures were built on which cohorts? -----------------------------
data("SignatureInfoTraining", package = "curatedTBData")
sit <- as.data.frame(SignatureInfoTraining)
stopifnot(all(c("TBSignature", "Study") %in% names(sit)))

# Our own conservative rule: treat the Berry discovery family (and its re-analyses) as one group.
berry_family <- c("GSE19435", "GSE19439", "GSE19442", "GSE19444",
                  "GSE107991", "GSE107992", "GSE107993", "GSE107994")

cat("TB-vs-LTBI signatures with a recorded training cohort:",
    sum(ltbi_sig %in% sit$TBSignature), "of", length(ltbi_sig), "\n")
cat("Origin unknown for:", paste(setdiff(ltbi_sig, sit$TBSignature), collapse = ", "), "\n")

# Sensitivity analysis: only TB-vs-LTBI signatures with KNOWN origin that were NOT built on GSE37250
clean_ids <- setdiff(unique(sit$TBSignature), sit$TBSignature[sit$Study == "GSE37250"])
sens <- bench[bench$Task == "TB vs LTBI" & bench$Signature %in% clean_ids, ]
print(sens[order(-sens$Gap), c("Signature", "AUC_Neg", "AUC_Pos", "Gap", "CI_low", "CI_high", "Clear_gap")],
      row.names = FALSE)
cat("\nSignatures:", nrow(sens), "| median gap:", round(median(sens$Gap), 3),
    "| gap above 0:", sum(sens$Gap > 0), "| clear (interval above 0):", sum(sens$Clear_gap), "\n")

TB-vs-LTBI signatures with a recorded training cohort: 13 of 32 
Origin unknown for: Blankley_5, Chen_5, Duffy_23, Estevez_133, Estevez_259, Gjoen_10, Gjoen_7, Jacobsen_3, Jenum_8, Kaul_3, Kwan_186, Li_3, Maertzdorf_15, Natarajan_7, Roe_3, Sivakumaran_11, Vargas_18, Vargas_42, Zhao_NANO_6 
    Signature AUC_Neg AUC_Pos    Gap CI_low CI_high Clear_gap
     Leong_24   0.947   0.861  0.086  0.026   0.148      TRUE
  Anderson_42   0.941   0.856  0.085  0.025   0.147      TRUE
 Blankley_380   0.950   0.874  0.076  0.017   0.135      TRUE
 Maertzdorf_4   0.938   0.868  0.070  0.009   0.133      TRUE
   Thompson_9   0.962   0.893  0.069  0.020   0.122      TRUE
    Berry_393   0.940   0.871  0.068  0.012   0.128      TRUE
    Walter_51   0.932   0.869  0.063  0.003   0.128      TRUE
    Esmail_82   0.942   0.884  0.058  0.002   0.115      TRUE
   Esmail_203   0.954   0.898  0.056  0.010   0.112      TRUE
  Tornheim_71   0.957   0.914  0.043 -0.007   0.095     FALSE
  Verhagen_10   0.728   0.7

In [8]:
## 7. OTHER COHORTS: GSE19442, GSE19444 (HIV-negative) and GSE69581 (HIV-positive) ---------
# Gene directions are learned in GSE37250 (TB vs LTBI) and applied unchanged. Only PTB and LTBI people used.
load_cohort <- function(acc) {
  dd <- curatedTBData(acc, dry.run = FALSE, curated.only = TRUE)[[acc]]
  e  <- assay(dd[["assay_curated"]]); m <- as.data.frame(colData(dd))
  stopifnot(all(colnames(e) == rownames(m)))
  cat(acc, ": genes x people", dim(e), "\n"); print(table(m$TBStatus, useNA = "ifany"))
  list(z = t(scale(t(e))), meta = m)
}
cohorts <- list(GSE19442 = load_cohort("GSE19442"),
                GSE19444 = load_cohort("GSE19444"),
                GSE69581 = load_cohort("GSE69581"))

use37 <- tb %in% c("PTB", "LTBI")
ext_tab <- do.call(rbind, lapply(ltbi_sig, function(s) {
  g0 <- usable_genes(s, rownames(z)); if (is.null(g0)) return(NULL)
  dirn <- sign(rowMeans(z[g0, use37 & is_tb == 1, drop = FALSE]) -
               rowMeans(z[g0, use37 & is_tb == 0, drop = FALSE]))
  sc1 <- colMeans(z[g0, , drop = FALSE] * dirn)
  hn <- use37 & hiv == "Negative"; hp <- use37 & hiv == "Positive"
  row <- data.frame(Signature = s,
                    GSE37250_HIVneg = round(auc_fast(is_tb[hn], sc1[hn]), 3),
                    GSE37250_HIVpos = round(auc_fast(is_tb[hp], sc1[hp]), 3))
  for (acc in names(cohorts)) {
    zz <- cohorts[[acc]]$z; mt <- cohorts[[acc]]$meta
    keep <- which(mt$TBStatus %in% c("PTB", "LTBI"))
    yy <- as.numeric(mt$TBStatus[keep] == "PTB")
    g <- g0[g0 %in% rownames(zz)]; g <- g[!is.na(rowSums(zz[g, , drop = FALSE]))]
    if (length(g) < 3 || length(g) < 0.8 * length(sigs[[s]])) { row[[acc]] <- NA; next }
    row[[acc]] <- round(auc_fast(yy, colMeans(zz[g, , drop = FALSE] * dirn[g])[keep]), 3)
  }
  row
}))
ext_tab$Origin_known <- ext_tab$Signature %in% unique(sit$TBSignature)

# Blank out each signature in every cohort it was built on
cohort_cols <- c("GSE37250_HIVneg", "GSE37250_HIVpos", "GSE19442", "GSE19444", "GSE69581")
linked <- list(GSE37250_HIVneg = "GSE37250", GSE37250_HIVpos = "GSE37250",
               GSE19442 = berry_family, GSE19444 = berry_family, GSE69581 = "GSE69581")
fair <- ext_tab
for (col in cohort_cols) {
  bad <- intersect(unique(sit$TBSignature[sit$Study %in% linked[[col]]]), fair$Signature)
  fair[fair$Signature %in% bad, col] <- NA
  cat(col, "- built on this cohort (removed):", if (length(bad)) paste(bad, collapse = ", ") else "none", "\n")
}
summ <- function(df) rbind(n = colSums(!is.na(df[, cohort_cols])),
                           median_AUC = round(sapply(df[, cohort_cols], median, na.rm = TRUE), 3))
cat("\nA) Non-circular signatures (origin-unknown kept):\n"); print(summ(fair))
cat("\nB) Non-circular AND origin known only:\n");           print(summ(fair[fair$Origin_known, ]))
cat("\nNote: cohorts differ in country, platform and case definition, and GSE69581 has no HIV-negative group.\n")

curated.only = TRUE. Download curated version. 
Set curated.only = FALSE if want to download both raw and curated data.

Downloading: GSE19442



  |=======================                                               |  33%
  |===============================================                       |  67%
  |======================================================================| 100%



Finished!



GSE19442 : genes x people 25417 51 

LTBI  PTB 
  31   20 


curated.only = TRUE. Download curated version. 
Set curated.only = FALSE if want to download both raw and curated data.

Downloading: GSE19444



  |=======================                                               |  33%
  |===============================================                       |  67%
  |======================================================================| 100%



... (2 outputs left out)

curated.only = TRUE. Download curated version. 
Set curated.only = FALSE if want to download both raw and curated data.

Downloading: GSE69581



  |=======================                                               |  33%
  |===============================================                       |  67%
  |======================================================================| 100%



Finished!



GSE69581 : genes x people 31467 50 

       LTBI         PTB Subclinical 
         25          15          10 
GSE37250_HIVneg - built on this cohort (removed): none 
GSE37250_HIVpos - built on this cohort (removed): none 
GSE19442 - built on this cohort (removed): Berry_393 
GSE19444 - built on this cohort (removed): Berry_393 
GSE69581 - built on this cohort (removed): Esmail_203, Esmail_82 

A) Non-circular signatures (origin-unknown kept):
           GSE37250_HIVneg GSE37250_HIVpos GSE19442 GSE19444 GSE69581
n                    28.00          28.000   25.000   25.000   26.000
median_AUC            0.95           0.871    0.952    0.868    0.901

B) Non-circular AND origin known only:
           GSE37250_HIVneg GSE37250_HIVpos GSE19442 GSE19444 GSE69581
n                   11.000          11.000    9.000    9.000    9.000
median_AUC           0.951           0.873    0.948    0.889    0.909

Note: cohorts differ in country, platform and case definition, and GSE69581 has no HIV-nega

In [9]:
## 8. EXPLORATORY MODELS (train in one country, test in the other) ---------------------------
# Elastic-net logistic regression, TB vs LTBI, 3000 most variable genes chosen in the training country.
#   baseline   : no adjustment
#   hiv_filter : remove genes that differ by HIV status among LTBI people in the training country
#   shift      : subtract the estimated HIV shift from HIV-positive people
#   random     : control; remove the same number of genes as hiv_filter, chosen at random
use_tl <- tb %in% c("PTB", "LTBI"); y <- is_tb

fit_predict <- function(train_c, test_c, mode = "baseline", seed = 1, n_remove = 0) {
  set.seed(seed)
  tr <- which(use_tl & country == train_c); te <- which(use_tl & country == test_c)
  genes <- names(sort(apply(expr[, tr], 1, var), decreasing = TRUE))[1:N_GENES]
  X <- expr[genes, , drop = FALSE]; removed <- 0
  if (mode == "hiv_filter") {
    l  <- tr[y[tr] == 0]
    pv <- sapply(genes, function(g) suppressWarnings(wilcox.test(expr[g, l] ~ hiv[l])$p.value))
    bad <- genes[p.adjust(pv, "BH") < 0.05]; removed <- length(bad); genes <- setdiff(genes, bad)
  } else if (mode == "random") {
    bad <- sample(genes, n_remove); removed <- length(bad); genes <- setdiff(genes, bad)
  } else if (mode == "shift") {
    l <- tr[y[tr] == 0]
    shift <- rowMeans(X[, l[hiv[l] == "Positive"], drop = FALSE]) -
             rowMeans(X[, l[hiv[l] == "Negative"], drop = FALSE])
    pos <- which(hiv == "Positive"); X[, pos] <- X[, pos] - shift
  }
  fit <- cv.glmnet(t(X[genes, tr, drop = FALSE]), y[tr], family = "binomial", alpha = 0.5, nfolds = 5)
  list(te = te,
       pr = as.numeric(predict(fit, t(X[genes, te, drop = FALSE]), s = "lambda.min", type = "link")),
       removed = removed,
       used = sum(as.numeric(coef(fit, s = "lambda.min"))[-1] != 0))
}
eval_pred <- function(p) {
  yt <- y[p$te]; h <- hiv[p$te]
  c(AUC_All = auc_fast(yt, p$pr),
    AUC_Neg = auc_fast(yt[h == "Negative"], p$pr[h == "Negative"]),
    AUC_Pos = auc_fast(yt[h == "Positive"], p$pr[h == "Positive"]))
}

dirs <- list(c("Malawi", "South Africa"), c("South Africa", "Malawi"))
model_rows <- list(); delta_rows <- list(); stab_rows <- list(); rand_rows <- list()

for (dr in dirs) {
  tr_c <- dr[1]; te_c <- dr[2]; cat("\n=== Train", tr_c, "-> test", te_c, "===\n")
  base <- fit_predict(tr_c, te_c, "baseline",   1)
  filt <- fit_predict(tr_c, te_c, "hiv_filter", 1)
  shft <- fit_predict(tr_c, te_c, "shift",      1)

  for (nm in c("baseline", "hiv_filter", "shift")) {
    p <- list(baseline = base, hiv_filter = filt, shift = shft)[[nm]]
    e <- eval_pred(p)
    model_rows[[length(model_rows) + 1]] <- data.frame(
      Train = tr_c, Test = te_c, Model = nm, Genes_removed = p$removed, Genes_used = p$used,
      AUC_All = round(e[1], 3), AUC_Neg = round(e[2], 3), AUC_Pos = round(e[3], 3),
      Gap = round(e[2] - e[3], 3), row.names = NULL)
  }

  # Bootstrap over test people: change versus baseline (overall AUC and HIV gap)
  yt <- y[base$te]; h <- hiv[base$te]; set.seed(2)
  bt <- replicate(B, {
    i <- sample(seq_along(yt), replace = TRUE)
    c(auc_fast(yt[i], filt$pr[i]) - auc_fast(yt[i], base$pr[i]),
      drop_of(yt[i], filt$pr[i], h[i]) - drop_of(yt[i], base$pr[i], h[i]),
      auc_fast(yt[i], shft$pr[i]) - auc_fast(yt[i], base$pr[i]),
      drop_of(yt[i], shft$pr[i], h[i]) - drop_of(yt[i], base$pr[i], h[i]))
  })
  labs <- expand.grid(Metric = c("overall AUC", "HIV gap"), Model = c("hiv_filter", "shift"), stringsAsFactors = FALSE)
  for (k in 1:4) {
    q <- q95(bt[k, ])
    delta_rows[[length(delta_rows) + 1]] <- data.frame(
      Train = tr_c, Test = te_c, Comparison = paste(labs$Model[k], "minus baseline"),
      Metric = labs$Metric[k], CI_low = round(q[1], 3), Median = round(q[2], 3), CI_high = round(q[3], 3))
  }

  # Stability over 10 seeds (baseline and filter)
  for (nm in c("baseline", "hiv_filter")) {
    a <- sapply(1:10, function(s) eval_pred(fit_predict(tr_c, te_c, nm, s))[1])
    stab_rows[[length(stab_rows) + 1]] <- data.frame(
      Train = tr_c, Test = te_c, Model = nm, Mean_AUC = round(mean(a), 3),
      Min = round(min(a), 3), Max = round(max(a), 3))
  }

  # Random-removal control: same number of genes as the HIV filter removed
  r <- sapply(1:20, function(s) eval_pred(fit_predict(tr_c, te_c, "random", s, n_remove = filt$removed))[1])
  rand_rows[[length(rand_rows) + 1]] <- data.frame(
    Train = tr_c, Test = te_c, Genes_removed = filt$removed,
    Mean_AUC = round(mean(r), 3), Min = round(min(r), 3), Max = round(max(r), 3))
}
model_tab <- do.call(rbind, model_rows); delta_tab <- do.call(rbind, delta_rows)
stab_tab  <- do.call(rbind, stab_rows);  rand_tab  <- do.call(rbind, rand_rows)
cat("\n--- Models (seed 1) ---\n");                         print(model_tab, row.names = FALSE)
cat("\n--- Change versus baseline (bootstrap) ---\n");      print(delta_tab, row.names = FALSE)
cat("\n--- Stability over 10 seeds ---\n");                 print(stab_tab,  row.names = FALSE)
cat("\n--- Random-removal control (20 repeats) ---\n");     print(rand_tab,  row.names = FALSE)


=== Train Malawi -> test South Africa ===

=== Train South Africa -> test Malawi ===

--- Models (seed 1) ---
        Train         Test      Model Genes_removed Genes_used AUC_All AUC_Neg
       Malawi South Africa   baseline             0         97   0.915   0.980
       Malawi South Africa hiv_filter           567         90   0.890   0.961
       Malawi South Africa      shift             0        100   0.912   0.987
 South Africa       Malawi   baseline             0        109   0.776   0.766
 South Africa       Malawi hiv_filter          1272         82   0.893   0.938
 South Africa       Malawi      shift             0        109   0.772   0.757
 AUC_Pos    Gap
   0.850  0.129
   0.840  0.121
   0.877  0.111
   0.796 -0.030
   0.900  0.037
   0.813 -0.056

--- Change versus baseline (bootstrap) ---
        Train         Test                Comparison      Metric CI_low Median
       Malawi South Africa hiv_filter minus baseline overall AUC -0.041 -0.024
       Malawi South Af

In [10]:
## 9. SAVE EVERYTHING ------------------------------------------------------------------------
dir.create("results", showWarnings = FALSE)
write.csv(bench,     "results/benchmark_gap_by_signature.csv", row.names = FALSE)
write.csv(ltbi_tab[, c("Signature", "Median_Neg", "Median_Pos", "p_raw", "p_adj", "Higher_in_HIV")],
          "results/ltbi_scores_by_hiv.csv", row.names = FALSE)
write.csv(sens,      "results/sensitivity_clean_signatures.csv", row.names = FALSE)
write.csv(ext_tab,   "results/other_cohorts_auc.csv", row.names = FALSE)
write.csv(model_tab, "results/models.csv", row.names = FALSE)
write.csv(delta_tab, "results/models_change_vs_baseline.csv", row.names = FALSE)
write.csv(stab_tab,  "results/models_seed_stability.csv", row.names = FALSE)
write.csv(rand_tab,  "results/models_random_control.csv", row.names = FALSE)
writeLines(capture.output(sessionInfo()), "results/sessionInfo.txt")
cat("Saved files:\n"); print(list.files("results"))

Saved files:
[1] "benchmark_gap_by_signature.csv"   "ltbi_scores_by_hiv.csv"          
[3] "models_change_vs_baseline.csv"    "models_random_control.csv"       
[5] "models_seed_stability.csv"        "models.csv"                      
[7] "other_cohorts_auc.csv"            "sensitivity_clean_signatures.csv"
[9] "sessionInfo.txt"                 


## Values to compare against (from the earlier exploratory runs)

If your numbers are close to these, the clean script reproduces the earlier work. Small differences are expected (B = 1000 instead of 500, one consistent 80 percent gene-coverage rule, and a fixed seed per model run).

| Section | What to expect (approximately) |
|---|---|
| 2 | 537 people, 31,537 genes. LTBI 83/84, OD 83/92, PTB 97/98 (HIV negative/positive) |
| 3 | AUC: everyone 0.897, HIV-negative 0.923, HIV-positive 0.877. Gap interval touches zero overall; clearer in South Africa than Malawi |
| 4 | About 46 signatures. Earlier: 24 with a clear gap, all but one TB-vs-LTBI. TB-vs-OD signatures: none clear |
| 5 | HIV-positive LTBI score higher in about 30 of 31 signatures; about 28 significant |
| 6 | Known-origin, not-built-on-GSE37250 TB-vs-LTBI signatures: 12; about 10 with a clear gap; median gap about 0.07 |
| 7 | Median AUC (Berry excluded): GSE37250 HIV-neg 0.95, GSE19442 0.95, GSE19444 0.87, GSE69581 0.90, GSE37250 HIV-pos 0.87 |
| 8 | Malawi to South Africa: baseline AUC about 0.915, gene filter about 0.89, random control about 0.916. South Africa to Malawi: baseline about 0.776, gene filter about 0.89, random control about 0.815. Shift correction: AUC about unchanged |

Known limits (also in the paper outline): one cohort and two countries; 19 signatures of unknown origin; no viral load or CD4 data; small external cohorts; signatures share genes, so "clear" counts are not independent evidence.